### Задание 1. Создание user-item-матрицы и разбиение данных на тест и контроль результатов

In [2]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
from pandas.api.types import CategoricalDtype
import implicit
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
ratings = pd.read_csv('ratings.csv')
books = pd.read_csv('books.csv')

In [3]:
# Преобразуем рейтинги в бинарные: 1 - книга понравилась, 0 - не понравилась
ratings['rating'] = ratings['rating'].apply(lambda x: 1 if x >= 4 else 0)

In [4]:
# Для каждого пользователя пронумеруем его прочитанные книги
ratings['book_order'] = ratings.groupby('user_id')['book_id'].rank(method='first')

In [5]:
# Рассчитаем доли прочитанных книг для каждого пользователя
total_books_per_user = ratings.groupby('user_id')['book_id'].count()
ratings = ratings.merge(total_books_per_user.rename('total_books'), on='user_id')
ratings['book_fraction'] = ratings['book_order'] / ratings['total_books']

In [6]:
# Разделим данные: 70% на обучение, 30% на контроль
ratings['is_train'] = ratings['book_fraction'] <= 0.7
train_ratings = ratings[ratings['is_train']]
test_ratings = ratings[~ratings['is_train']]
# Почему не стоит использовать стандартные методы разделения выборки?
# Случайное разделение (train_test_split): Было бы нереалистично, так как в реальности система не знает, какие книги пользователь прочитает в будущем.
# K-fold кросс-валидация: Тоже не подходит, так как нарушает временную последовательность и может привести к утечке информации.
# Стратифицированное разделение: Не имеет смысла для рекомендательных систем, где важен порядок взаимодействия пользователя с предметами.
# Мы здесь сделали разделение по порядку прочтения, система учится на ранних предпочтениях пользователя и тестируется на способности предсказывать более поздние предпочтения, что имитирует реальную ситуацию рекомендаций.

In [7]:
# Создаем user-item матрицу для данных обучения
user_index = train_ratings['user_id'].unique()
book_index = train_ratings['book_id'].unique()

rows = train_ratings['user_id'].astype(CategoricalDtype(categories=user_index)).cat.codes
cols = train_ratings['book_id'].astype(CategoricalDtype(categories=book_index)).cat.codes

train_matrix = sparse.csr_matrix((train_ratings['rating'], (rows, cols)), 
                                  shape=(len(user_index), len(book_index)))

In [ ]:
# Проверяем размеры матрицы
print(f"Количество пользователей: {train_matrix.shape[0]}")
print(f"Количество книг: {train_matrix.shape[1]}")
print(f"Уникальных пользователей в исходных данных: {len(ratings['user_id'].unique())}")
print(f"Уникальных книг в исходных данных: {len(ratings['book_id'].unique())}")
# Кол-во книг - это кол-во в user-item матрице, где применена фильтрация по рейтингу 4+ и выделена обучающая выборка, поэтому их меньше, чем уникальных книг в исходных данных, пусть это не удивляет

Количество пользователей: 53424
Количество книг: 6654
Уникальных пользователей в исходных данных: 53424
Уникальных книг в исходных данных: 10000


In [9]:
# Выбираем случайного пользователя для демонстрации
sample_user_id = user_index[0]

# Находим книги с высокой оценкой для этого пользователя
user_books = train_ratings[train_ratings['user_id'] == sample_user_id]
high_rated_books = user_books[user_books['rating'] == 1]

# Соединяем с информацией о книгах
high_rated_books = high_rated_books.merge(books, left_on='book_id', right_on='book_id')

print(f"Книги с высокой оценкой для пользователя {sample_user_id}:")
print(high_rated_books[['book_id', 'title', 'authors']])

Книги с высокой оценкой для пользователя 1:
    book_id                                              title  \
0       258  The Shadow of the Wind (The Cemetery of Forgot...   
1        11                                    The Kite Runner   
2       136             Divine Secrets of the Ya-Ya Sisterhood   
3        35                                      The Alchemist   
4        33                                Memoirs of a Geisha   
5        10                                Pride and Prejudice   
6         4                              To Kill a Mockingbird   
7        70                    Ender's Game (Ender's Saga, #1)   
8        36                          The Giver (The Giver, #1)   
9        32                                    Of Mice and Men   
10       13                                               1984   
11       66                                 Gone with the Wind   
12       43                                          Jane Eyre   
13       45                     

In [10]:
# Сохраняем индекс пользователя для использования в следующих заданиях
sample_user_index = np.where(user_index == sample_user_id)[0][0]
print(f"Индекс пользователя {sample_user_id} в матрице: {sample_user_index}")

Индекс пользователя 1 в матрице: 0


### Задание 2. Создание бейзлайнов и расчёт метрик

In [35]:
def average_precision_at_k(recommended_items, relevant_items, k=10):
    """Вычисляет Average Precision at K"""
    relevant_items = set(relevant_items)
    if not relevant_items:
        return 0.0
    
    score = 0.0
    num_hits = 0.0
    
    # Используем min(k, len(recommended_items)) для корректной обрезки
    k = min(k, len(recommended_items))
    
    for i in range(k):
        if recommended_items[i] in relevant_items:
            num_hits += 1.0
            # Правильный расчет precision: num_hits / (i + 1)
            precision = num_hits / (i + 1)
            score += precision

    # Проверяем, есть ли вообще релевантные книги в рекомендациях
    if num_hits == 0:
        return 0.0
    return score / num_hits



In [38]:
import random

# Фиксируем список из 500 случайных пользователей
random.seed(42)
sample_users = random.sample(list(user_index), 500)

def generate_random_recommendations(user_id, k=10):
    # Только книги, прочитанные пользователем В ОБУЧЕНИИ
    user_train_books = train_ratings[train_ratings['user_id'] == user_id]['book_id'].unique()
    # Все книги, доступные для рекомендаций (из обучающей матрицы)
    all_books = set(book_index)  # book_index содержит book_id из train_ratings
    unread_books = list(all_books - set(user_train_books))
    return random.sample(unread_books, min(k, len(unread_books)))

random_recommendations = {}
for user_id in sample_users:
    random_recommendations[user_id] = generate_random_recommendations(user_id)

In [39]:
# Рассчитываем mAP@10 для случайного бейзлайна
map_scores = []
for user_id in sample_users:
    # Книги, прочитанные пользователем в обучении
    train_user_books = train_ratings[train_ratings['user_id'] == user_id]['book_id'].unique()
    
    # Релевантные книги из теста: рейтинг 1 И не прочитаны в обучении
    relevant_books = test_ratings[(test_ratings['user_id'] == user_id) & 
                                  (test_ratings['rating'] == 1) &
                                  (~test_ratings['book_id'].isin(train_user_books))]['book_id'].unique()
    
    if len(relevant_books) == 0:
        continue
    
    recommendations = random_recommendations[user_id][:10]
    map_score = average_precision_at_k(recommendations, relevant_books, k=10)
    map_scores.append(map_score)

if len(map_scores) == 0:
    print("Нет пользователей с релевантными книгами в тестовой выборке!")
else:
    random_baseline_map = np.mean(map_scores)
    print(f"mAP@10 для случайного бейзлайна: {random_baseline_map:.4f}")

mAP@10 для случайного бейзлайна: 0.0072


In [40]:
# Популярность считаем ТОЛЬКО на обучающих данных
book_popularity = train_ratings.groupby('book_id').size().reset_index(name='count')
popular_books = book_popularity.sort_values('count', ascending=False)['book_id'].tolist()

def generate_popular_recommendations(user_id, k=10):
    # Книги, прочитанные пользователем в обучении
    user_train_books = train_ratings[train_ratings['user_id'] == user_id]['book_id'].unique()
    # Все книги из обучающей матрицы
    all_books = set(book_index)
    unread_books = list(all_books - set(user_train_books))
    
    # Самые популярные среди непрочитанных
    popular_unread = [book_id for book_id in popular_books if book_id in unread_books]
    return popular_unread[:k]

popular_recommendations = {}
for user_id in sample_users:
    popular_recommendations[user_id] = generate_popular_recommendations(user_id)

In [41]:
# Рассчитываем mAP@10 для бейзлайна популярных книг
map_scores = []
for user_id in sample_users:
    train_user_books = train_ratings[train_ratings['user_id'] == user_id]['book_id'].unique()
    
    relevant_books = test_ratings[(test_ratings['user_id'] == user_id) & 
                                  (test_ratings['rating'] == 1) &
                                  (~test_ratings['book_id'].isin(train_user_books))]['book_id'].unique()
    
    if len(relevant_books) == 0:
        continue
    
    recommendations = popular_recommendations[user_id][:10]
    map_score = average_precision_at_k(recommendations, relevant_books, k=10)
    map_scores.append(map_score)

popular_baseline_map = np.mean(map_scores)
print(f"mAP@10 для бейзлайна популярных книг: {popular_baseline_map:.4f}")

mAP@10 для бейзлайна популярных книг: 0.0000


Результат не удивителен. Бейзлайн «популярные айтемы» принципиально слаб для персонализированной задачи. Он предлагает одинаковый набор книг всем пользователям. На реальных данных часто даёт mAP, близкий к нулю.

### Задание 3. Применение метода матричной факторизации и улучшение параметров алгоритма факторизации

In [44]:
# Создаем модель ALS с базовыми параметрами
model = implicit.als.AlternatingLeastSquares(factors=64, iterations=30, 
                                            calculate_training_loss=True)

# Обучаем модель
model.fit(train_matrix)

100%|██████████| 30/30 [00:05<00:00,  5.81it/s, loss=0.00529]


In [47]:
# Получаем рекомендации для нашего примера пользователя
user_items = train_matrix[sample_user_index].toarray().reshape(-1)
ids, scores = model.recommend(sample_user_index, 
                              sparse.csr_matrix(user_items), 
                              N=20, 
                              filter_already_liked_items=True)

print(f"Рекомендации для пользователя {sample_user_id}:")
for book_id, score in zip(ids, scores):
    book_info = books[books['book_id'] == book_index[book_id]]
    if not book_info.empty:
        print(f"ID: {book_id}, Оценка: {score:.4f}, Название: {book_info['title'].values[0]}")

Рекомендации для пользователя 1:
ID: 50, Оценка: 1.3417, Название: The Book Thief
ID: 1522, Оценка: 0.8679, Название: A Thousand Splendid Suns
ID: 287, Оценка: 0.6488, Название: The Secret Life of Bees
ID: 96, Оценка: 0.5765, Название: One Hundred Years of Solitude
ID: 94, Оценка: 0.5317, Название: Sense and Sensibility
ID: 2972, Оценка: 0.5104, Название: The Guernsey Literary and Potato Peel Pie Society
ID: 190, Оценка: 0.5086, Название: The Red Tent
ID: 272, Оценка: 0.4707, Название: The Little Prince
ID: 231, Оценка: 0.4616, Название: Wuthering Heights
ID: 289, Оценка: 0.4509, Название: Anna Karenina
ID: 398, Оценка: 0.4076, Название: The Time Traveler's Wife
ID: 35, Оценка: 0.4047, Название: Animal Farm
ID: 325, Оценка: 0.3966, Название: The Joy Luck Club
ID: 171, Оценка: 0.3818, Название: The Old Man and the Sea
ID: 79, Оценка: 0.3812, Название: The Curious Incident of the Dog in the Night-Time
ID: 122, Оценка: 0.3732, Название: Girl with a Pearl Earring
ID: 95, Оценка: 0.3724, На

In [49]:
# Рассчитываем mAP@10 для ALS модели
map_scores = []
for user_id in sample_users:
    user_index_in_matrix = np.where(user_index == user_id)[0][0]
    
    train_user_books = train_ratings[train_ratings['user_id'] == user_id]['book_id'].unique()
    relevant_books = test_ratings[(test_ratings['user_id'] == user_id) & 
                                  (test_ratings['rating'] == 1) &
                                  (~test_ratings['book_id'].isin(train_user_books))]['book_id'].unique()
    
    if len(relevant_books) == 0:
        continue
    
    user_items = train_matrix[user_index_in_matrix].toarray().reshape(-1)
    ids, _ = model.recommend(user_index_in_matrix, 
                             sparse.csr_matrix(user_items), 
                             N=20, 
                             filter_already_liked_items=True)
    
    recommended_book_ids = [book_index[book_id] for book_id in ids[:10]]
    map_score = average_precision_at_k(recommended_book_ids, relevant_books, k=10)
    map_scores.append(map_score)

als_map = np.mean(map_scores)
print(f"mAP@10 для ALS модели с базовыми параметрами: {als_map:.4f}")

mAP@10 для ALS модели с базовыми параметрами: 0.0037


In [51]:
models = {
    'model_1': {'factors': 128, 'iterations': 30},
    'model_2': {'factors': 64, 'iterations': 50},
    'model_3': {'factors': 128, 'iterations': 50},
    'model_4': {'factors': 256, 'iterations': 30}
}

results = {}

for model_name, params in models.items():
    print(f"Обучение модели {model_name} с параметрами: factors={params['factors']}, iterations={params['iterations']}")
    
    model = implicit.als.AlternatingLeastSquares(factors=params['factors'], 
                                                iterations=params['iterations'],
                                                calculate_training_loss=True)
    model.fit(train_matrix)
    
    map_scores = []
    for user_id in sample_users:
        user_index_in_matrix = np.where(user_index == user_id)[0][0]
        
        # Книги, прочитанные пользователем в обучении
        train_user_books = train_ratings[train_ratings['user_id'] == user_id]['book_id'].unique()
        
        # Релевантные книги из теста, которые пользователь не читал в обучении
        relevant_books = test_ratings[(test_ratings['user_id'] == user_id) & 
                                      (test_ratings['rating'] == 1) &
                                      (~test_ratings['book_id'].isin(train_user_books))]['book_id'].unique()
        
        if len(relevant_books) == 0:
            continue
        
        # Получаем вектор оценок пользователя (только обучение)
        user_items = train_matrix[user_index_in_matrix].toarray().reshape(-1)
        
        # Рекомендации модели — распаковываем ids и scores
        ids, _ = model.recommend(user_index_in_matrix, 
                                 sparse.csr_matrix(user_items), 
                                 N=20, 
                                 filter_already_liked_items=True)
        
        # Преобразуем внутренние индексы книг в реальные book_id (первые 10)
        recommended_book_ids = [book_index[book_id] for book_id in ids[:10]]
        
        # Рассчитываем AP@10
        map_score = average_precision_at_k(recommended_book_ids, relevant_books, k=10)
        map_scores.append(map_score)
    
    results[model_name] = {
        'map': np.mean(map_scores) if map_scores else 0.0,
        'factors': params['factors'],
        'iterations': params['iterations']
    }
    
    print(f"mAP@10 для модели {model_name}: {results[model_name]['map']:.4f}")

Обучение модели model_1 с параметрами: factors=128, iterations=30


100%|██████████| 30/30 [00:07<00:00,  3.76it/s, loss=0.00455]


mAP@10 для модели model_1: 0.0076
Обучение модели model_2 с параметрами: factors=64, iterations=50


100%|██████████| 50/50 [00:08<00:00,  6.04it/s, loss=0.00529]


mAP@10 для модели model_2: 0.0029
Обучение модели model_3 с параметрами: factors=128, iterations=50


100%|██████████| 50/50 [00:12<00:00,  4.03it/s, loss=0.00454]


mAP@10 для модели model_3: 0.0061
Обучение модели model_4 с параметрами: factors=256, iterations=30


100%|██████████| 30/30 [03:14<00:00,  6.50s/it, loss=0.00361]


mAP@10 для модели model_4: 0.0087


In [52]:
print("\nСравнение моделей:")
print("-" * 60)
print(f"{'Модель':<15} {'Factors':<10} {'Iterations':<12} {'mAP@10':<10}")
print("-" * 60)

for model_name, result in results.items():
    print(f"{model_name:<15} {result['factors']:<10} {result['iterations']:<12} {result['map']:<10.4f}")

print("\nСравнение с бейзлайнами:")
print(f"Случайный бейзлайн: {random_baseline_map:.4f}")
print(f"Бейзлайн популярных книг: {popular_baseline_map:.4f}")

for model_name, result in results.items():
    if result['map'] > popular_baseline_map:
        print(f"Модель {model_name} превзошла бейзлайн популярных книг!")
    else:
        print(f"Модель {model_name} не превзошла бейзлайн популярных книг.")


Сравнение моделей:
------------------------------------------------------------
Модель          Factors    Iterations   mAP@10    
------------------------------------------------------------
model_1         128        30           0.0076    
model_2         64         50           0.0029    
model_3         128        50           0.0061    
model_4         256        30           0.0087    

Сравнение с бейзлайнами:
Случайный бейзлайн: 0.0072
Бейзлайн популярных книг: 0.0000
Модель model_1 превзошла бейзлайн популярных книг!
Модель model_2 превзошла бейзлайн популярных книг!
Модель model_3 превзошла бейзлайн популярных книг!
Модель model_4 превзошла бейзлайн популярных книг!


## Выводы о качестве модели

- **Бейзлайны**  
  - Случайные рекомендации: **mAP@10 = 0.0072**  
  - Популярные книги: **mAP@10 = 0.0000** (неперсонализированный подход не работает на данных)

- **Модели ALS**  
  - Лучший результат: **model\_4** (factors=256, iterations=30) — **mAP@10 = 0.0087**  
  - Остальные модели:  
    - model\_1 (128,30): 0.0076  
    - model\_3 (128,50): 0.0061  
    - model\_2 (64,50): 0.0029  
  - **Все модели превзошли бейзлайн популярных книг** (цель достигнута).  

- **Итог**  
  - ALS позволяет персонализировать рекомендации и превзойти неинформированный популярный бейзлайн, но **абсолютные значения mAP@10 остаются низкими (<0.01)**.  
  - Это может быть связано с высокой разреженностью данных, бинарным представлением рейтингов или особенностями временного разделения.  
  - Для дальнейшего улучшения качества необходима настройка регуляризации, использование весов взаимодействий или добавление признаков (авторы, жанры).